# Lab 2 — Data Quality

Practiced data-quality checks on the week3 orders data:

* Schema validation with `DESCRIBE QUERY` and `_rescued_data` before loading anything
* Bronze → silver → gold layering with lineage columns (`source_file`, `ingestion_time`)
* Null filtering and `TRY_CAST` for safe bad-data detection
* `EXPECT` constraints in the pipeline: `ON VIOLATION DROP ROW` vs `FAIL UPDATE`

In [0]:
%sql
-- Preview the raw orders JSON before loading anything.
-- _metadata.file_name records WHICH source file each row came from (lineage); current_time() marks when it was ingested
select *,_metadata.file_name as source_file,current_time() as ingestion_time from read_files(
    "/Volumes/databricks_labs/default/lab_data/week3/orders/",
    format => "json"
)

In [0]:
%sql
-- Data-quality first step: dry-run the query to see the inferred schema WITHOUT reading all the files.
-- Note _rescued_data — malformed records are parked there instead of failing the load
DESCRIBE QUERY 
select *,_metadata.file_name as source_file,current_time() as ingestion_time from read_files(
    "/Volumes/databricks_labs/default/lab_data/week3/orders/",
    format => "json"
) LIMIT 0;


In [0]:
%sql
-- BRONZE layer: land the raw JSON as-is, adding the two lineage columns (source_file, ingestion_time)
CREATE OR REPLACE TABLE orders_bronze AS(
    select *,_metadata.file_name as source_file,current_timestamp() as ingestion_time from read_files(
    "/Volumes/databricks_labs/default/lab_data/week3/orders/",
    format => "json"
)
);

SELECT * FROM orders_bronze;

In [0]:
%sql
-- Inspect the bronze schema: original JSON columns + _rescued_data + the two lineage columns we added
DESCRIBE TABLE orders_bronze;

In [0]:
%sql
-- SILVER layer (data-quality step): cast order_timestamp to TIMESTAMP, derive order_date,
-- and DROP any row with a null order_id or customer_id so bad records never reach gold
CREATE or REPLACE  TABLE orders_silver AS(
    SELECT customer_id,notifications,order_id,CAST(order_timestamp AS TIMESTAMP),source_file,ingestion_time,to_date(CAST(order_timestamp AS TIMESTAMP)) as order_date 
    FROM orders_bronze
    where order_id is not null and customer_id is not null
);

select * from orders_silver;

In [0]:
%sql
-- GOLD layer: aggregate silver into a daily summary view — total orders and unique customers per order_date
CREATE OR REPLACE VIEW orders_gold AS(
    select order_date,count(order_id) as total_orders,count(distinct customer_id) as unique_customers
    from orders_silver
    group by order_date
    order by order_date
);

select * from orders_gold

In [0]:
%sql
-- Same bronze layer, but built as a STREAMING TABLE by the declarative pipeline (orders_bronze_st) — compare with the batch table above
select * from databricks_labs.default.orders_bronze_st

In [0]:
%sql
-- Row count in silver after the pipeline's EXPECT data-quality constraint ran (invalid rows were dropped)
SELECT COUNT(*) AS silver_count
FROM databricks_labs.default.orders_silver_st;

In [0]:
%sql
-- End-to-end reconciliation: record counts should line up across all three layers (Bronze = Silver = Gold total)
SELECT 'Bronze' AS layer, COUNT(*) AS records
FROM databricks_labs.default.orders_bronze_st

UNION ALL

SELECT 'Silver', COUNT(*)
FROM databricks_labs.default.orders_silver_st

UNION ALL

SELECT 'Gold', SUM(total_orders)
FROM databricks_labs.default.orders_gold_st;

In [0]:
%sql
-- TRY_CAST for quality checks: bad values become NULL instead of throwing an error (regular CAST would fail) — safe way to detect dirty data
SELECT TRY_CAST('100$' AS INT);

In [0]:
%sql
-- EXPECT ... ON VIOLATION FAIL UPDATE: when a row violates the constraint the WHOLE pipeline update fails,
-- so the target streaming table stays empty (0 rows) — nothing is partially committed
-- what happens when fail update condition comes true
SELECT COUNT(*) AS total_records
FROM databricks_labs.default.orders_silver_fail_st;

In [0]:
%sql
-- Clean up: drop the bronze table created to demo the FAIL UPDATE constraint
DROP TABLE databricks_labs.default.orders_bronze_fail_st;

In [0]:
%sql
-- Clean up: drop the silver fail-demo table as well
DROP TABLE databricks_labs.default.orders_silver_fail_st;